In [1]:
import ast
import re
import time
import requests
import logging as log
from collections import deque
from pathlib import Path
from typing import Any
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urldefrag
from neo4j import GraphDatabase

log.basicConfig(level=log.INFO, format='%(message)s')

In [2]:
"""
Unified PyTorch 2.x documentation scraper for The Structural Coder.

Pipeline (4 phases):
  Phase 0  Seed PyTorchConcept nodes (TorchDynamo, Inductor, etc.)
  Phase 1  Parse index.html toctree → pre-seed BFS queue with all known
           API pages (no random discovery — we know every URL upfront)
  Phase 2  BFS crawl over the seeded queue, extracting Concepts +
           CodeSnippets + CALLS edges, writing to Neo4j live
  Phase 3  Structured parse of API pages: typed Class/Function/Method/
           Parameter nodes + INHERITS / HAS_PARAM edges
  Phase 4  Wire REPLACES edges (modern → legacy idiom mapping)

Neo4j labels produced:
    API_Endpoint   — a docs page URL
    Concept        — explanatory paragraph text
    CodeSnippet    — a docs code block with torch.* calls
    API_Function   — any torch.* symbol (from CALLS or structured parse)
    API_Class      — typed class entity from structured parse
    API_Method     — method within a class
    API_Parameter  — formal parameter of a callable
    DeprecatedAPI  — a legacy 1.x symbol
    PyTorchConcept — semantic 2.x concept (Dynamo, Inductor, etc.)

Edge types produced:
    REFERENCES  — API_Endpoint → API_Endpoint  (toctree / BFS links)
    EXPLAINS    — API_Endpoint → Concept
    IMPLEMENTS  — API_Endpoint → CodeSnippet
    CALLS       — CodeSnippet  → API_Function
    CONTAINS    — API_Endpoint → API_Class / API_Function
    INHERITS    — API_Class    → API_Class
    HAS_PARAM   — API_Function / API_Method → API_Parameter
    REPLACES    — API_Function → DeprecatedAPI
    RELATED_TO  — API_Function → PyTorchConcept
    SAME_AS     — API_Function → API_Class  (dedup between BFS+structured)
"""

DOCS_BASE = "https://docs.pytorch.org/docs/stable/"

_SESSION_HEADERS = {
    "User-Agent": (
        "StructuralCoder-KGBuilder/1.0 "
        "(research project; graph-augmented code generation)"
    )
}

# Modules to run the structured API parse on 
# Paths verified directly from the real index.html toctree (2.10).

STRUCTURED_MODULES: list[dict] = [
    # Core tensor API
    {"q": "torch",                              "doc": "torch"},
    {"q": "torch.Tensor",                       "doc": "tensors"},
    {"q": "torch.nn",                           "doc": "nn"},
    {"q": "torch.nn.functional",               "doc": "nn.functional"},
    {"q": "torch.nn.init",                      "doc": "nn.init"},
    {"q": "torch.nn.attention",                "doc": "nn.attention",                        "new": True},
    {"q": "torch.optim",                        "doc": "optim"},
    {"q": "torch.autograd",                     "doc": "autograd"},
    # 2.x compile stack
    {"q": "torch.compile",                      "doc": "generated/torch.compile",            "new": True},
    {"q": "torch._dynamo",                      "doc": "torch.compiler_api",                 "new": True},  
    {"q": "torch.export",                       "doc": "user_guide/torch_compiler/export",   "new": True},  
    {"q": "torch.func",                         "doc": "func",                                "new": True},
    # Mixed precision
    {"q": "torch.amp",                          "doc": "amp",                                "new": True},
    # Distributed
    {"q": "torch.distributed",                  "doc": "distributed"},
    {"q": "torch.distributed.fsdp",             "doc": "fsdp",                               "new": True},
    {"q": "torch.distributed.fsdp.fully_shard", "doc": "distributed.fsdp.fully_shard",       "new": True},
    {"q": "torch.distributed.tensor",           "doc": "distributed.tensor"},
    {"q": "torch.distributed.tensor.parallel",  "doc": "distributed.tensor.parallel"},
    {"q": "torch.distributed.optim",            "doc": "distributed.optim"},
    {"q": "torch.distributed.pipelining",       "doc": "distributed.pipelining",             "new": True},
    {"q": "torch.distributed.checkpoint",       "doc": "distributed.checkpoint"},
    # Data & profiling
    {"q": "torch.utils.data",                   "doc": "data"},
    {"q": "torch.profiler",                     "doc": "profiler"},
    {"q": "torch.utils.benchmark",              "doc": "benchmark_utils"},
    # Device backends
    {"q": "torch.cuda",                         "doc": "cuda"},
    {"q": "torch.backends",                     "doc": "backends"},
    # Math / signal
    {"q": "torch.linalg",                       "doc": "linalg"},
    {"q": "torch.fft",                          "doc": "fft"},
    {"q": "torch.special",                      "doc": "special"},
    # FX / JIT
    {"q": "torch.fx",                           "doc": "fx"},
    {"q": "torch.jit",                          "doc": "jit"},
    # Misc
    {"q": "torch.library",                      "doc": "library",                            "new": True},
    {"q": "torch.testing",                      "doc": "testing"},
    {"q": "torch.sparse",                       "doc": "sparse"},
]

# ── Legacy → modern idiom REPLACES edges ──────────────────────────────────────
PYTORCH_2X_IDIOMS: dict[str, str] = {
    "torch.compile":                                     "torch.jit.script",
    "torch.compile":                                     "torch.jit.trace",
    "torch.nn.functional.scaled_dot_product_attention":  "manual_attention_impl",
    "torch.amp.autocast":                                "torch.cuda.amp.autocast",
    "torch.distributed.fsdp.FullyShardedDataParallel":   "torch.nn.DataParallel",
    "torch.utils.data.StatefulDataLoader":               "torch.utils.data.DataLoader",
    "torch._dynamo":                                     "torch.fx",
    "torch.export":                                      "torch.jit.export",
}

# ── Semantic concept nodes ─────────────────────────────────────────────────────
PYTORCH_CONCEPTS: list[dict] = [
    {"name": "TorchDynamo",   "desc": "Python bytecode capture frontend for torch.compile"},
    {"name": "TorchInductor", "desc": "Default 2.x compiler backend; emits Triton/C++ kernels"},
    {"name": "TritonBackend", "desc": "GPU kernel synthesis backend used by TorchInductor"},
    {"name": "AOTAutograd",   "desc": "Ahead-of-time autograd tracing for compile"},
    {"name": "FSDP2",         "desc": "Fully Sharded Data Parallel v2 — distributed training"},
    {"name": "FlexAttention", "desc": "Composable attention API (PyTorch 2.3+)"},
    {"name": "CudaGraphs",    "desc": "CUDA graph capture mode for torch.compile"},
]

_CONCEPT_MAP: dict[str, str] = {
    "torch.compile":      "TorchDynamo",
    "torch._dynamo":      "TorchDynamo",
    "torch.amp.autocast": "AOTAutograd",
    "torch.export":       "TorchDynamo",
}

# Toctree link labels that are NOT torch API pages — skip structured parse
_NON_API_LABELS: set[str] = {
    "User Guide", "PyTorch Main Components", "Torch.compile", "Torch.export",
    "Developer Notes", "Accelerator Integration", "Reference API",
    "Automatic Mixed Precision examples", "Autograd mechanics",
    "Broadcasting semantics", "CPU threading and TorchScript inference",
    "CUDA semantics", "PyTorch Custom Operators Landing Page",
    "Distributed Data Parallel", "Extending PyTorch",
    "Extending torch.func with autograd.Function", "Frequently Asked Questions",
    "Getting Started on Intel GPU", "Gradcheck mechanics", "HIP (ROCm) semantics",
    "Features for large-scale deployments", "LibTorch Stable ABI",
    "MKLDNN backend", "Bfloat16 (BF16) on MKLDNN backend", "Modules",
    "MPS backend", "Multiprocessing best practices", "Numerical accuracy",
    "Out Notes", "Reproducibility", "Serialization semantics", "Windows FAQ",
    "Community", "PyTorch Governance | Build + CI", "PyTorch Contribution Guide",
    "PyTorch Design Philosophy", "PyTorch Governance | Mechanics",
    "PyTorch Governance | Maintainers",
}

In [ ]:
# Main scraper class

class Scraper:
    """ Unified PyTorch docs scraper → Neo4j knowledge graph. """

    def __init__(
        self,
        index_html: str,            # path to local index.html (or a URL)
        uri: str,
        user: str,
        password: str,
        crawl_delay: float = 0.4,
        max_crawl_pages: int | None = None,
    ):
        self.index_html = index_html
        self.crawl_delay = crawl_delay
        self.max_crawl_pages = max_crawl_pages

        self.queue:   deque[str] = deque()
        self.visited: set[str] = set()

        self._session = requests.Session()
        self._session.headers.update(_SESSION_HEADERS)

        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self) -> None:
        self.driver.close()


    # Top-level orchestration

    def run(self) -> None:
        log.info("=== Phase 0: Seed PyTorchConcept nodes ===")
        self._seed_concepts()

        log.info("=== Phase 1: Parse toctree → seed BFS queue ===")
        seeded = self._seed_from_toctree(self.index_html)
        log.info("Seeded %d URLs from toctree.", seeded)

        log.info("=== Phase 2: BFS crawl ===")
        self._bfs_crawl()

        log.info("=== Phase 3: Structured API parse ===")
        self._structured_parse_all()

        log.info("=== Phase 4: Wire REPLACES edges ===")
        self._wire_replaces_edges()

        log.info("Scraper finished.")


    # Phase 0 — concept seed

    def _seed_concepts(self) -> None:
        with self.driver.session() as session:
            for c in PYTORCH_CONCEPTS:
                session.execute_write(
                    lambda tx, c=c: tx.run(
                        """
                        MERGE (pc:PyTorchConcept {name: $name})
                        ON CREATE SET pc.description = $desc, pc.version = '2.x'
                        """,
                        name=c["name"], desc=c["desc"],
                    )
                )
        log.info("Seeded %d PyTorchConcept nodes.", len(PYTORCH_CONCEPTS))


    # Phase 1 — toctree-driven BFS seed

    def _seed_from_toctree(self, source: str) -> int:
        """
        Parse the toctree of index.html and pre-load the BFS queue with
        every 'reference internal' link that lives under DOCS_BASE.

        `source` can be:
          - a local file path  (str or Path)
          - a full https:// URL (fetched with the session)

        Returns the number of unique URLs added to the queue.
        """
        if source.startswith("http"):
            try:
                resp = self._session.get(source, timeout=15)
                resp.raise_for_status()
                html = resp.text
            except requests.RequestException as exc:
                log.error("Could not fetch index: %s", exc)
                return 0
        else:
            html = Path(source).read_text(encoding="utf-8", errors="replace")

        soup = BeautifulSoup(html, "html.parser")

        # toctree links that stay on docs.pytorch.org/docs/stable/
        count = 0
        for a in soup.select("div.toctree-wrapper a.reference.internal"):
            href = a.get("href", "")
            if not href or href.startswith("#"):
                continue

            # Strip fragment — we want the page, not the section
            href_clean, _ = urldefrag(href)

            # Resolve relative to DOCS_BASE (index.html is at root)
            full_url = urljoin(DOCS_BASE, href_clean)

            if full_url not in self.visited and full_url.startswith(DOCS_BASE):
                self.queue.append(full_url)
                self.visited.add(full_url)
                count += 1

        # Also seed the index page itself
        index_url = DOCS_BASE + "index.html"
        if index_url not in self.visited:
            self.queue.appendleft(index_url)
            self.visited.add(index_url)
            count += 1

        return count


    # Phase 2 — BFS crawler

    def _bfs_crawl(self) -> None:
        pages_crawled = 0
        while self.queue:
            if self.max_crawl_pages and pages_crawled >= self.max_crawl_pages:
                log.info("Reached max_crawl_pages=%d, stopping BFS.", self.max_crawl_pages)
                break

            current_url = self.queue.popleft()
            log.info("[BFS %d] %s", pages_crawled + 1, current_url)

            new_links, page_data = self.process_page(current_url)

            if page_data:
                with self.driver.session() as session:
                    session.execute_write(self._build_graph_topology, page_data, new_links)

            for link in new_links:
                if link not in self.visited:
                    self.visited.add(link)
                    self.queue.append(link)

            pages_crawled += 1
            time.sleep(self.crawl_delay)

    def process_page(self, url: str) -> tuple[list[str], dict | None]:
        """Fetch + clean a page; extract concepts, code snippets, and links."""
        try:
            res = self._session.get(url, timeout=10)
            res.raise_for_status()
        except requests.RequestException as exc:
            log.warning("Failed to fetch %s: %s", url, exc)
            return [], None

        soup = BeautifulSoup(res.content, "html.parser")

        # Strip chrome 
        for tag in soup.find_all(["nav", "footer", "script", "style"]):
            tag.decompose()
        for div in soup.find_all(
            "div",
            class_=["bd-sidebar-primary", "bd-sidebar-secondary", "cookie-banner-wrapper"],
        ):
            div.decompose()

        raw_title   = soup.title.string if soup.title else "Unknown"
        clean_title = re.sub(
            r"\s*[—–-]\s*PyTorch\s+\d.*?documentation.*$", "", raw_title
        ).strip()

        page_data: dict[str, Any] = {
            "url":           url,
            "title":         clean_title or "Unknown",
            "concepts":      [],
            "code_snippets": [],
        }

        article = soup.find("article", class_="bd-article")
        if article:
            # Concepts: every meaningful paragraph
            for p in article.find_all("p"):
                text = p.get_text(strip=True)
                if len(text) > 30:
                    page_data["concepts"].append(text)

            # Code blocks with torch references
            for block in article.find_all("div", class_="highlight"):
                code = block.get_text(strip=True)
                if "torch." in code or "import torch" in code:
                    page_data["code_snippets"].append({
                        "code":  code,
                        "calls": self.extract_torch_calls(code),
                    })

        # Discover new links within stable docs 
        new_links: list[str] = []
        for a_tag in soup.find_all("a", href=True):
            full_url  = urljoin(url, a_tag["href"])
            clean_url, _ = urldefrag(full_url)
            if clean_url.startswith(DOCS_BASE):
                new_links.append(clean_url)

        return new_links, page_data


    # AST helpers 

    @staticmethod
    def _get_full_call_name(node: ast.expr) -> str | None:
        """Recursively build the full dotted name from an AST call node."""
        if isinstance(node, ast.Name):
            return node.id
        if isinstance(node, ast.Attribute):
            base = Scraper._get_full_call_name(node.value) 
            if base:
                return f"{base}.{node.attr}"
        return None

    @staticmethod
    def extract_torch_calls(code_string: str) -> list[str]:
        """Return all torch.* function names called in a code snippet."""
        calls: set[str] = set()
        try:
            tree = ast.parse(code_string)
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    name = Scraper._get_full_call_name(node.func)
                    if name and name.startswith("torch."):
                        calls.add(name)
        except SyntaxError:
            pass   # Sphinx snippets are often partial — silently skip
        return list(calls)

    # Neo4j write: BFS topology 

    @staticmethod
    def _build_graph_topology(
        tx: Any,
        page_data: dict,
        new_links: list[str],
    ) -> None:
        url   = page_data["url"]
        title = page_data["title"]

        tx.run(
            "MERGE (api:API_Endpoint {url: $url}) ON CREATE SET api.title = $title",
            url=url, title=title,
        )
        for concept in page_data["concepts"]:
            tx.run(
                """
                MATCH (api:API_Endpoint {url: $url})
                MERGE (c:Concept {text: $concept})
                MERGE (api)-[:EXPLAINS]->(c)
                """,
                url=url, concept=concept,
            )
        for s in page_data["code_snippets"]:
            tx.run(
                """
                MATCH (api:API_Endpoint {url: $url})
                MERGE (cs:CodeSnippet {code: $code})
                MERGE (api)-[:IMPLEMENTS]->(cs)
                """,
                url=url, code=s["code"],
            )
            if s["calls"]:
                tx.run(
                    """
                    MATCH (cs:CodeSnippet {code: $code})
                    UNWIND $calls AS fn_name
                    MERGE (fn:API_Function {name: fn_name})
                    MERGE (cs)-[:CALLS]->(fn)
                    """,
                    code=s["code"], calls=s["calls"],
                )
        for link in new_links:
            tx.run(
                """
                MATCH (api:API_Endpoint {url: $url})
                MERGE (t:API_Endpoint {url: $target})
                MERGE (api)-[:REFERENCES]->(t)
                """,
                url=url, target=link,
            )


    # Phase 3 — Structured API parse

    def _structured_parse_all(self) -> None:
        for mod in STRUCTURED_MODULES:
            url    = urljoin(DOCS_BASE, mod["doc"] + ".html")
            is_new = mod.get("new", False)
            log.info("[Structured] %s → %s", mod["q"], url)
            try:
                resp = self._session.get(url, timeout=10)
                resp.raise_for_status()
            except requests.RequestException as exc:
                log.warning("Structured parse failed for %s: %s", mod["q"], exc)
                time.sleep(self.crawl_delay)
                continue

            soup = BeautifulSoup(resp.text, "html.parser")
            with self.driver.session() as session:
                for dl in soup.find_all("dl", class_=re.compile(r"\bpy\s+class\b")):
                    session.execute_write(self._ingest_class, dl, mod["q"], url, is_new)
                for dl in soup.find_all("dl", class_=re.compile(r"\bpy\s+function\b")):
                    session.execute_write(self._ingest_function, dl, mod["q"], url, is_new)

            time.sleep(self.crawl_delay)

    @staticmethod
    def _ingest_class(tx: Any, dl: Any, module_q: str, page_url: str, is_new: bool) -> None:
        dt = dl.find("dt")
        if not dt:
            return
        name = Scraper._extract_sig_name(dt)
        if not name:
            return
        qualified = f"{module_q}.{name}"
        version   = "2.0" if is_new else "1.x"

        tx.run(
            """
            MERGE (cls:API_Class {qualified: $q})
            SET cls.name = $name, cls.module = $mod,
                cls.docstring = $doc, cls.version = $ver,
                cls.source_url = $url
            """,
            q=qualified, name=name, mod=module_q,
            doc=Scraper._extract_first_docstring(dl)[:400],
            ver=version, url=page_url,
        )
        tx.run(
            """
            MATCH (api:API_Endpoint {url: $url})
            MATCH (cls:API_Class {qualified: $q})
            MERGE (api)-[:CONTAINS]->(cls)
            """,
            url=page_url, q=qualified,
        )
        # Deduplicate with API_Function node that BFS may have created via CALLS
        tx.run(
            """
            MATCH (cls:API_Class {qualified: $q})
            OPTIONAL MATCH (fn:API_Function {name: $q})
            WITH cls, fn WHERE fn IS NOT NULL
            MERGE (fn)-[:SAME_AS]->(cls)
            """,
            q=qualified,
        )
        # Inheritance
        bases_tag = dl.find(string=re.compile(r"Bases?:"))
        if bases_tag:
            for a in bases_tag.find_next_siblings("a"):
                base_text = a.get_text(strip=True)
                base_q    = base_text if "." in base_text else f"{module_q}.{base_text}"
                tx.run(
                    """
                    MERGE (base:API_Class {qualified: $bq})
                    ON CREATE SET base.name = $bn
                    WITH base
                    MATCH (cls:API_Class {qualified: $q})
                    MERGE (cls)-[:INHERITS]->(base)
                    """,
                    bq=base_q, bn=base_q.split(".")[-1], q=qualified,
                )
        # Methods
        for method_dl in dl.find_all("dl", class_=re.compile(r"\bpy\s+method\b")):
            Scraper._ingest_method(tx, method_dl, qualified, page_url)

    @staticmethod
    def _ingest_function(tx: Any, dl: Any, module_q: str, page_url: str, is_new: bool) -> None:
        dt = dl.find("dt")
        if not dt:
            return
        name = Scraper._extract_sig_name(dt)
        if not name:
            return
        qualified = f"{module_q}.{name}"
        version   = "2.0" if is_new else "1.x"

        # MERGE: BFS may have already created this node via CALLS extraction
        tx.run(
            """
            MERGE (fn:API_Function {name: $q})
            SET fn.qualified = $q, fn.module = $mod,
                fn.docstring = $doc, fn.version = $ver,
                fn.source_url = $url
            """,
            q=qualified, mod=module_q,
            doc=Scraper._extract_first_docstring(dl)[:400],
            ver=version, url=page_url,
        )
        tx.run(
            """
            MATCH (api:API_Endpoint {url: $url})
            MATCH (fn:API_Function {name: $q})
            MERGE (api)-[:CONTAINS]->(fn)
            """,
            url=page_url, q=qualified,
        )
        Scraper._ingest_params(tx, dt, qualified)

    @staticmethod
    def _ingest_method(tx: Any, dl: Any, class_q: str, page_url: str) -> None:
        dt = dl.find("dt")
        if not dt:
            return
        name = Scraper._extract_sig_name(dt)
        if not name:
            return
        qualified = f"{class_q}.{name}"
        tx.run(
            """
            MERGE (m:API_Method {qualified: $q})
            SET m.name = $name, m.docstring = $doc, m.source_url = $url
            WITH m
            MATCH (cls:API_Class {qualified: $cq})
            MERGE (cls)-[:CONTAINS]->(m)
            """,
            q=qualified, name=name,
            doc=Scraper._extract_first_docstring(dl)[:400],
            url=page_url, cq=class_q,
        )
        Scraper._ingest_params(tx, dt, qualified)

    @staticmethod
    def _ingest_params(tx: Any, dt: Any, parent_q: str) -> None:
        for em in dt.find_all("em", class_="sig-param"):
            raw  = em.get_text(strip=True)
            name = raw.split(":")[0].split("=")[0].strip().lstrip("*")
            if not name or not name.isidentifier():
                continue
            param_q = f"{parent_q}.{name}"
            tx.run(
                """
                MERGE (p:API_Parameter {qualified: $pq})
                SET p.name = $name, p.parent = $parent
                WITH p
                MATCH (parent {qualified: $parent})
                MERGE (parent)-[:HAS_PARAM]->(p)
                """,
                pq=param_q, name=name, parent=parent_q,
            )


    # Phase 4 — REPLACES edges

    def _wire_replaces_edges(self) -> None:
        with self.driver.session() as session:
            for modern_q, legacy_name in PYTORCH_2X_IDIOMS.items():
                session.execute_write(self._create_replaces_edge, modern_q, legacy_name)

    @staticmethod
    def _create_replaces_edge(tx: Any, modern_q: str, legacy_name: str) -> None:
        tx.run(
            """
            MERGE (dep:DeprecatedAPI {name: $legacy})
            ON CREATE SET dep.note = 'Legacy PyTorch 1.x — use modern equivalent'
            """,
            legacy=legacy_name,
        )
        tx.run(
            """
            MERGE (fn:API_Function {name: $modern})
            ON CREATE SET fn.qualified = $modern
            WITH fn
            MATCH (dep:DeprecatedAPI {name: $legacy})
            MERGE (fn)-[:REPLACES {weight: 1.5}]->(dep)
            """,
            modern=modern_q, legacy=legacy_name,
        )
        if modern_q in _CONCEPT_MAP:
            tx.run(
                """
                MATCH (fn:API_Function {name: $modern})
                MATCH (pc:PyTorchConcept {name: $concept})
                MERGE (fn)-[:RELATED_TO]->(pc)
                """,
                modern=modern_q, concept=_CONCEPT_MAP[modern_q],
            )

    # Shared HTML helpers

    @staticmethod
    def _extract_sig_name(dt: Any) -> str:
        code = dt.find("code", class_="sig-name")
        if code:
            return code.get_text(strip=True)
        id_attr = dt.get("id", "")
        return id_attr.split(".")[-1] if id_attr else ""

    @staticmethod
    def _extract_first_docstring(section: Any) -> str:
        dd = section.find("dd")
        if dd:
            p = dd.find("p")
            if p:
                return p.get_text(strip=True)
        return ""

In [4]:
if __name__ == "__main__":
    scraper = Scraper(
        index_html="index.html",
        uri="bolt://localhost:7687", 
        user="neo4j", 
        password="pytorch2026"
    )
    scraper.run() 
    scraper.close()

=== Phase 0: Seed PyTorchConcept nodes ===
Seeded 7 PyTorchConcept nodes.
=== Phase 1: Parse toctree → seed BFS queue ===
Seeded 120 URLs from toctree.
=== Phase 2: BFS crawl ===
[BFS 1] https://docs.pytorch.org/docs/stable/index.html
[BFS 2] https://docs.pytorch.org/docs/stable/user_guide/index.html
[BFS 3] https://docs.pytorch.org/docs/stable/user_guide/pytorch_main_components.html
[BFS 4] https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/torch.compiler.html
[BFS 5] https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/export.html
[BFS 6] https://docs.pytorch.org/docs/stable/notes.html
[BFS 7] https://docs.pytorch.org/docs/stable/accelerator/index.html
[BFS 8] https://docs.pytorch.org/docs/stable/pytorch-api.html
[BFS 9] https://docs.pytorch.org/docs/stable/torch.html
[BFS 10] https://docs.pytorch.org/docs/stable/nn.html
[BFS 11] https://docs.pytorch.org/docs/stable/nn.functional.html
[BFS 12] https://docs.pytorch.org/docs/stable/tensors.html
[BFS 13] https://do